# Anomaly Detection for Energy Analytics - NSP Dataset

## Objective
Implement multiple anomaly detection methods and export results for Tableau dashboard.

Required output: `anomaly_results.csv` with columns:
- timestamp
- region
- consumption_kwh
- anomaly_flag
- anomaly_type
- anomaly_method

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Paths
ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Helper function
def save_and_show_plot(fig, filename):
    output_path = OUTPUT_DIR / filename
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {output_path}")
    plt.show()

print("Setup complete!")

In [ ]:
# Load engineered features
df = pd.read_csv(OUTPUT_DIR / 'engineered_features.csv', parse_dates=['timestamp'])

# Reconstruct region column if needed
if 'region' not in df.columns:
    region_cols = [col for col in df.columns if col.startswith('region_')]
    df['region'] = df[region_cols].idxmax(axis=1).str.replace('region_', '')

print(f"Dataset shape: {df.shape}")
print(f"Regions: {sorted(df['region'].unique())}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

# Check existing anomaly_flag column
if 'anomaly_flag' in df.columns:
    print(f"\nExisting anomaly_flag distribution:")
    print(df['anomaly_flag'].value_counts())
    print(f"Anomaly rate: {df['anomaly_flag'].sum() / len(df) * 100:.2f}%")

df.head()

## Understanding Existing Anomalies

Let's analyze the pre-labeled anomalies in the dataset before implementing our own detection methods.

In [ ]:
# Analyze existing anomalies by region
anomaly_by_region = df.groupby('region')['anomaly_flag'].agg(['sum', 'count', 'mean'])
anomaly_by_region.columns = ['Anomaly_Count', 'Total_Records', 'Anomaly_Rate']
anomaly_by_region['Anomaly_Rate'] = anomaly_by_region['Anomaly_Rate'] * 100

print("Existing Anomalies by Region:")
print(anomaly_by_region)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot of counts
anomaly_by_region['Anomaly_Count'].plot(kind='bar', ax=axes[0], color='coral')
axes[0].set_title('Anomaly Count by Region')
axes[0].set_ylabel('Number of Anomalies')
axes[0].set_xlabel('Region')
axes[0].tick_params(axis='x', rotation=45)

# Anomaly rate
anomaly_by_region['Anomaly_Rate'].plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Anomaly Rate by Region (%)')
axes[1].set_ylabel('Anomaly Rate (%)')
axes[1].set_xlabel('Region')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
save_and_show_plot(fig, 'existing_anomalies_by_region.png')

## Statistical Method 1: Z-Score Detection

Detects anomalies based on standard deviations from the mean.
- Threshold: |z-score| > 3
- Applied per region to account for regional differences

In [ ]:
def detect_zscore_anomalies(df, column='consumption_kwh', threshold=3):
    """Detect anomalies using Z-score method per region"""
    df_copy = df.copy()
    df_copy['zscore_anomaly'] = 0
    
    for region in df_copy['region'].unique():
        mask = df_copy['region'] == region
        values = df_copy.loc[mask, column]
        
        # Calculate z-scores
        z_scores = np.abs(stats.zscore(values, nan_policy='omit'))
        
        # Mark anomalies
        df_copy.loc[mask, 'zscore_anomaly'] = (z_scores > threshold).astype(int)
    
    return df_copy

# Apply Z-score detection
df = detect_zscore_anomalies(df)

zscore_stats = df.groupby('region')['zscore_anomaly'].agg(['sum', 'mean'])
zscore_stats.columns = ['Anomaly_Count', 'Anomaly_Rate']
zscore_stats['Anomaly_Rate'] = zscore_stats['Anomaly_Rate'] * 100

print("Z-Score Anomaly Detection Results:")
print(zscore_stats)
print(f"\nTotal anomalies detected: {df['zscore_anomaly'].sum()} ({df['zscore_anomaly'].mean()*100:.2f}%)")

## Statistical Method 2: IQR (Interquartile Range) Detection

Robust to outliers, uses quartile-based thresholds.
- Anomalies: values < Q1 - 1.5×IQR or > Q3 + 1.5×IQR
- Applied per region

In [ ]:
def detect_iqr_anomalies(df, column='consumption_kwh', multiplier=1.5):
    """Detect anomalies using IQR method per region"""
    df_copy = df.copy()
    df_copy['iqr_anomaly'] = 0
    
    for region in df_copy['region'].unique():
        mask = df_copy['region'] == region
        values = df_copy.loc[mask, column]
        
        # Calculate IQR
        Q1 = values.quantile(0.25)
        Q3 = values.quantile(0.75)
        IQR = Q3 - Q1
        
        # Define bounds
        lower_bound = Q1 - multiplier * IQR
        upper_bound = Q3 + multiplier * IQR
        
        # Mark anomalies
        anomalies = (values < lower_bound) | (values > upper_bound)
        df_copy.loc[mask, 'iqr_anomaly'] = anomalies.astype(int)
    
    return df_copy

# Apply IQR detection
df = detect_iqr_anomalies(df)

iqr_stats = df.groupby('region')['iqr_anomaly'].agg(['sum', 'mean'])
iqr_stats.columns = ['Anomaly_Count', 'Anomaly_Rate']
iqr_stats['Anomaly_Rate'] = iqr_stats['Anomaly_Rate'] * 100

print("IQR Anomaly Detection Results:")
print(iqr_stats)
print(f"\nTotal anomalies detected: {df['iqr_anomaly'].sum()} ({df['iqr_anomaly'].mean()*100:.2f}%)")

## Machine Learning Method: Isolation Forest

Uses multivariate features to detect anomalies:
- Features: consumption_kwh, lags, rolling stats, weather, time features
- Contamination rate: 0.02 (expect ~2% anomalies, matching existing labels)
- Applied per region

In [ ]:
def detect_isolation_forest_anomalies(df, contamination=0.02):
    """Detect anomalies using Isolation Forest with multivariate features"""
    df_copy = df.copy()
    df_copy['isolation_forest_anomaly'] = 0
    
    # Select features for anomaly detection
    feature_cols = ['consumption_kwh', 'consumption_kwh_lag_1h', 'consumption_kwh_lag_24h',
                   'rolling_mean_168h', 'rolling_std_24h',
                   'temperature_c', 'humidity_pct', 'hour', 'day_of_week',
                   'grid_load_pct', 'renewable_pct']
    
    for region in df_copy['region'].unique():
        print(f"Training Isolation Forest for {region}...")
        mask = df_copy['region'] == region
        
        # Extract features and drop NaN
        X = df_copy.loc[mask, feature_cols].ffill().bfill()
        
        # Standardize features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Train Isolation Forest
        iso_forest = IsolationForest(contamination=contamination, 
                                    random_state=42,
                                    n_estimators=100)
        predictions = iso_forest.fit_predict(X_scaled)
        
        # Convert predictions: -1 (anomaly) -> 1, 1 (normal) -> 0
        anomalies = (predictions == -1).astype(int)
        df_copy.loc[mask, 'isolation_forest_anomaly'] = anomalies
    
    return df_copy

# Apply Isolation Forest
df = detect_isolation_forest_anomalies(df, contamination=0.02)

iso_stats = df.groupby('region')['isolation_forest_anomaly'].agg(['sum', 'mean'])
iso_stats.columns = ['Anomaly_Count', 'Anomaly_Rate']
iso_stats['Anomaly_Rate'] = iso_stats['Anomaly_Rate'] * 100

print("\nIsolation Forest Anomaly Detection Results:")
print(iso_stats)
print(f"\nTotal anomalies detected: {df['isolation_forest_anomaly'].sum()} ({df['isolation_forest_anomaly'].mean()*100:.2f}%)")

## Method Comparison

Comparing detection rates across all methods.

In [ ]:
# Create comparison summary
comparison = pd.DataFrame({
    'Method': ['Existing Labels', 'Z-Score', 'IQR', 'Isolation Forest'],
    'Total_Anomalies': [
        df['anomaly_flag'].sum(),
        df['zscore_anomaly'].sum(),
        df['iqr_anomaly'].sum(),
        df['isolation_forest_anomaly'].sum()
    ],
    'Anomaly_Rate_%': [
        df['anomaly_flag'].mean() * 100,
        df['zscore_anomaly'].mean() * 100,
        df['iqr_anomaly'].mean() * 100,
        df['isolation_forest_anomaly'].mean() * 100
    ]
})

print("\nMethod Comparison:")
print(comparison.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
comparison.plot(x='Method', y='Anomaly_Rate_%', kind='bar', ax=ax, color='teal', legend=False)
ax.set_title('Anomaly Detection Rate by Method')
ax.set_ylabel('Anomaly Rate (%)')
ax.set_xlabel('Detection Method')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
save_and_show_plot(fig, 'anomaly_detection_comparison.png')

## Export Anomaly Results for Tableau

Creating unified anomaly results with all detection methods.

In [ ]:
# Create unified anomaly flag and type
df['unified_anomaly_flag'] = (
    (df['anomaly_flag'] == 1) | 
    (df['zscore_anomaly'] == 1) | 
    (df['iqr_anomaly'] == 1) | 
    (df['isolation_forest_anomaly'] == 1)
).astype(int)

# Determine anomaly type/method
def get_anomaly_methods(row):
    methods = []
    if row['anomaly_flag'] == 1:
        methods.append('Existing')
    if row['zscore_anomaly'] == 1:
        methods.append('Z-Score')
    if row['iqr_anomaly'] == 1:
        methods.append('IQR')
    if row['isolation_forest_anomaly'] == 1:
        methods.append('IsolationForest')
    return '|'.join(methods) if methods else 'Normal'

df['anomaly_method'] = df.apply(get_anomaly_methods, axis=1)

# Get anomaly_type from existing column or set to 'Detected'
if 'anomaly_type' not in df.columns:
    df['anomaly_type'] = 'Detected'

# Create Tableau export
anomaly_results = df[['timestamp', 'region', 'consumption_kwh', 
                      'unified_anomaly_flag', 'anomaly_type', 'anomaly_method']].copy()
anomaly_results.columns = ['timestamp', 'region', 'consumption_kwh', 
                           'anomaly_flag', 'anomaly_type', 'anomaly_method']

# Save
anomaly_results.to_csv(OUTPUT_DIR / 'anomaly_results.csv', index=False)

print(f"✓ Exported anomaly_results.csv")
print(f"  Total rows: {len(anomaly_results)}")
print(f"  Anomalies flagged: {anomaly_results['anomaly_flag'].sum()} ({anomaly_results['anomaly_flag'].mean()*100:.2f}%)")
print(f"\nSample:")
print(anomaly_results[anomaly_results['anomaly_flag'] == 1].head(10))

## Summary and Recommendations

### Detection Methods Implemented:

1. **Z-Score Method** (0.77% detection rate)
   - Most conservative, flags only extreme outliers
   - Best for: Identifying severe consumption spikes

2. **IQR Method** (1.51% detection rate)
   - Moderate sensitivity, robust to distribution
   - Best for: General outlier detection

3. **Isolation Forest** (2.00% detection rate)
   - ML-based, considers multiple features
   - Best for: Complex multivariate anomalies

### Key Findings:
- **Total unique anomalies detected**: 17,716 (4.04% of dataset)
- **Overlap between methods**: Multiple methods flagging same points indicates high confidence
- **Existing labels validation**: 9,052 pre-labeled anomalies (2.07%) serve as ground truth

### Export for Tableau:
- File: `output/anomaly_results.csv`
- Contains: All detected anomalies with detection method labels
- Ready for: Dashboard visualization and comparison

### Recommendations:
1. **High-confidence anomalies**: Points flagged by 2+ methods
2. **Investigation priority**: Isolation Forest + IQR overlap
3. **Model comparison**: Use with forecast_results.csv to analyze prediction errors at anomaly points

In [ ]:
# Final overlap statistics
if 'isolation_anomaly' not in df.columns and 'isolation_forest_anomaly' in df.columns:
    df['isolation_anomaly'] = df['isolation_forest_anomaly']

print("Method Overlap Analysis:")
print(f"Flagged by all 3 methods: {((df['zscore_anomaly']==1) & (df['iqr_anomaly']==1) & (df['isolation_anomaly']==1)).sum()}")
print(f"Flagged by 2+ methods: {((df['zscore_anomaly'] + df['iqr_anomaly'] + df['isolation_anomaly']) >= 2).sum()}")
print(f"Flagged by only 1 method: {((df['zscore_anomaly'] + df['iqr_anomaly'] + df['isolation_anomaly']) == 1).sum()}")

# High-confidence anomalies
df['high_confidence_anomaly'] = ((df['zscore_anomaly'] + df['iqr_anomaly'] + df['isolation_anomaly']) >= 2).astype(int)

print(f"\nHigh-confidence anomalies (2+ methods): {df['high_confidence_anomaly'].sum()}")
print("\nBy region:")
print(df.groupby('region')['high_confidence_anomaly'].sum())

## Machine Learning Method: Isolation Forest

Tree-based anomaly detection using multiple features.
- Uses consumption + lag features + rolling stats
- Contamination parameter: 0.02 (expected 2% anomalies)
- Applied per region